In [0]:
from azure.storage.blob import BlobServiceClient
from pyspark.sql.functions import col, decode, split, element_at,udf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType
from pyspark.sql.functions import date_format, col, when, count, lit, concat, round as spark_round, regexp_extract, expr
import time
import logging

In [0]:
logger = logging.getLogger("DatabricksWorkflow")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)

In [0]:
dbutils.fs.rm("dbfs:/checkpoints/APPEALS/ARCHIVE/", recurse=True)
dbutils.fs.rm("dbfs:/checkpoints/ARCHIVE/", recurse=True)

In [0]:
#Load configuration JSON
config_path = "dbfs:/configs/config.json"
try:
    config = spark.read.option("multiline", "true").json(config_path)
    logger.info(f"Successfully read config file from {config_path}")
except Exception as e:
    logger.error(f"Could not read config file at {config_path}: {e}", exc_info=True)
    raise FileNotFoundError(f"Could not read config file at {config_path}: {e}")

#Extract environment and lz_key
try:
    first_row = config.first()
    env = first_row["env"].strip().lower()
    lz_key = first_row["lz_key"].strip().lower()
    logger.info(f"Extracted configs: env={env}, lz_key={lz_key}")
except Exception as e:
    logger.error(f"Missing expected keys 'env' or 'lz_key' in config file: {e}", exc_info=True)
    raise KeyError(f"Missing expected keys 'env' or 'lz_key' in config file: {e}")

#Construct keyvault name
try:
    keyvault_name = f"ingest{lz_key}-meta002-{env}"
    logger.info(f"Constructed keyvault name: {keyvault_name}")
except Exception as e:
    logger.error(f"Error constructing keyvault name: {e}", exc_info=True)
    raise ValueError(f"Error constructing keyvault name: {e}")


In [0]:
# Access the Service Principal secrets from Key Vault
try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-SECRET from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-SECRET' from Key Vault '{keyvault_name}': {e}")

try:
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-TENANT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-TENANT-ID' from Key Vault '{keyvault_name}': {e}")

try:
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    logger.info("Successfully retrieved SERVICE-PRINCIPLE-CLIENT-ID from Key Vault")
except Exception as e:
    logger.error(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}", exc_info=True)
    raise KeyError(f"Could not retrieve 'SERVICE-PRINCIPLE-CLIENT-ID' from Key Vault '{keyvault_name}': {e}")

logger.info("✅ Successfully retrieved all Service Principal secrets from Key Vault")


In [0]:
# --- Parameterise containers ---
curated_storage_account = f"ingest{lz_key}curated{env}"
curated_container = "gold"
silver_curated_container = "silver"
checkpoint_storage_account = f"ingest{lz_key}xcutting{env}"

# --- Assign OAuth to storage accounts ---
storage_accounts = [curated_storage_account, checkpoint_storage_account]

for storage_account in storage_accounts:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }

        for key, val in configs.items():
            try:
                spark.conf.set(key, val)
            except Exception as e:
                logger.error(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}", exc_info=True)
                raise RuntimeError(f"Failed to set Spark config '{key}' for storage account '{storage_account}': {e}")

        logger.info(f"✅ Successfully configured OAuth for storage account: {storage_account}")

    except Exception as e:
        logger.error(f"Error configuring OAuth for storage account '{storage_account}': {e}", exc_info=True)
        raise RuntimeError(f"Error configuring OAuth for storage account '{storage_account}': {e}")


In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAJR-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAJR/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"IU files: {uk_count}")
print(f"ARIAJR/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAJR-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAJR/submission"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAUTA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAUTA/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"IU files: {uk_count}")
print(f"ARIAUTA/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAUTA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAUTA/submission"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAFTA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAFTA/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"IU files: {uk_count}")
print(f"ARIAFTA/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAFTA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAFTA/submission"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAFPA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAFPA/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"UK files: {uk_count}")
print(f"ARIAB/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAFPA-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAFPA/submission"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIASB-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIASB/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"UK files: {uk_count}")
print(f"ARIASB/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIASB-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIASB/collected"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAB-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAB/response"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

# dbutils.fs.ls(input_path1)

cr_files = []
uf_files = []
uk_files = []

for x in dbutils.fs.ls(input_path1):
    if x.name.endswith("_cr.rsp"):
        cr_files.append(x.path)
    elif x.name.endswith("_uf.rsp"):
        uf_files.append(x.path)
    else: uk_files.append(x.path)

# counts
cr_count = len(cr_files)
uf_count = len(uf_files)
uk_count = len(uk_files)

print(f"CR files: {cr_count}")
print(f"UF files: {uf_count}")
print(f"UK files: {uk_count}")
print(f"ARIAB/response count {cr_count+uf_count+uk_count}")

In [0]:
sas_token = dbutils.secrets.get(
    scope=keyvault_name,
    key=f"ARIAB-SAS-TOKEN"
)

input_path1 = f"wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAB/collected"
spark.conf.set(f"fs.azure.sas.dropzone.a360c2x2555dz.blob.core.windows.net", sas_token)

len(dbutils.fs.ls(input_path1))

In [0]:
df = spark.read.option("multiline", "false").json("wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAJR/collected/judicial_officer_1.a360i31c6c2f064fe7bb35357bb3acec10da6")
df.display()

In [0]:
from pyspark.sql.functions import (
    col,
    expr,
    input_file_name,
    from_unixtime
)

# -------------------------------------------------
# 1. Ensure correct timezone (fix BST/GMT issue)
# -------------------------------------------------
spark.conf.set("spark.sql.session.timeZone", "Europe/London")

# -------------------------------------------------
# 2. Base path (no dbutils looping for reading)
# -------------------------------------------------
base_path = "wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAJR/collected/"

# -------------------------------------------------
# 3. Read all files as text
# -------------------------------------------------
df = spark.read.text(base_path + "*")

# -------------------------------------------------
# 4. Capture source file path per row
# -------------------------------------------------
df = df.withColumn("source_path", input_file_name())

# -------------------------------------------------
# 5. Get file metadata once (dbutils side)
# -------------------------------------------------
files = dbutils.fs.ls(base_path)

df_files = spark.createDataFrame(files).select(
    col("path"),
    col("modificationTime")
)

# -------------------------------------------------
# 6. Join file metadata onto content
# -------------------------------------------------
df = df.join(
    df_files,
    df.source_path == df_files.path,
    "left"
)

# -------------------------------------------------
# 7. Filter last 3 hours (incremental load window)
# -------------------------------------------------
df_filtered = df.filter(
    col("modificationTime") >= expr(
        "unix_timestamp(current_timestamp() - INTERVAL 3 HOURS) * 1000"
    )
)

# -------------------------------------------------
# 8. Convert to readable timestamp (BST-aware)
# -------------------------------------------------
df_filtered = df_filtered.withColumn(
    "readable_time",
    from_unixtime(col("modificationTime") / 1000)
)

# -------------------------------------------------
# 9. Final output
# -------------------------------------------------
df_filtered.select(
    "source_path",
    "value",
    "modificationTime",
    "readable_time"
).display()

In [0]:
base_path = "wasbs://dropzone@a360c2x2555dz.blob.core.windows.net/ARIAJR/collected/*"

df = spark.read.text(base_path)